Пример кода DAG

In [ ]:
from airflow import DAG
from airflow.operators.python_operator import PythonOperator
from datetime import datetime

default_args = {
    'owner': 'airflow',
    'start_date': datetime(2024, 9, 1),
}

def print_execution_date(execution_date, **kwargs):
    print(f"Execution date: {execution_date}")

def process_data(**kwargs):
    execution_date = kwargs['execution_date']
    # Используем XCOM для передачи результата
    kwargs['ti'].xcom_push(key='execution_date', value=str(execution_date))

def save_data(**kwargs):
    execution_date = kwargs['ti'].xcom_pull(task_ids='process_data_task', key='execution_date')
    print(f"Saved data for execution date: {execution_date}")

with DAG('dag_with_macros_xcom', default_args=default_args, schedule_interval='@daily') as dag:
    print_execution_date_task = PythonOperator(task_id='print_execution_date_task',
                                               python_callable=print_execution_date,
                                               provide_context=True,
                                               op_args=['{{ ds }}'])

    process_data_task = PythonOperator(task_id='process_data_task',
                                       python_callable=process_data,
                                       provide_context=True)

    save_data_task = PythonOperator(task_id='save_data_task',
                                    python_callable=save_data,
                                    provide_context=True)

    print_execution_date_task >> process_data_task >> save_data_task
